# Week 4 – End-to-End ML Project Walkthrough (Titanic survival)
This notebook walks through the project step by step: **data → preprocessing pipeline → model comparison → saved model → API call**.
The reproducible training code lives in `ml/train.py`; the notebook explains and demonstrates it.

In [1]:
import sys, json, time, pickle, tempfile, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, joblib
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from ml.features import add_features, RAW_FEATURES
from ml.train import build_pipeline, load_data
pd.set_option("display.max_columns", 30)

## 1. Load and inspect the raw data

In [2]:
raw = pd.read_csv(ROOT / "data" / "titanic.csv")
print(raw.shape)
raw.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
miss = raw.isnull().sum(); print(miss[miss > 0])
print("\nSurvival rate:", round(raw["Survived"].mean(), 3))
print(raw.groupby("Sex")["Survived"].mean().round(3).to_dict(), raw.groupby("Pclass")["Survived"].mean().round(3).to_dict())

Age         177
Cabin       687
Embarked      2
dtype: int64

Survival rate: 0.384
{'female': 0.742, 'male': 0.189} {1: 0.63, 2: 0.473, 3: 0.242}


`Age` (20% missing), `Cabin` (77% missing) and `Embarked` (2 missing) need handling. `Cabin` is reduced to a simple yes/no flag `HasCabin`. All other cleaning happens **inside the pipeline**, so exactly the same steps are applied to new data in the API.

## 2. Features and the pipeline

In [4]:
X, y = load_data(ROOT / "data" / "titanic.csv")
print("Model inputs:", list(X.columns))
before = X.head(3)
after = add_features(before)
print("\nAfter feature engineering (new: FamilySize, IsAlone; Fare capped at 250):")
after

Model inputs: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'HasCabin']

After feature engineering (new: FamilySize, IsAlone; Fare capped at 250):


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,FamilySize,IsAlone
0,3,male,22.0,1,0,7.2500,S,0,2,0
1,1,female,38.0,1,0,71.2833,C,1,2,0
2,3,female,26.0,0,0,7.9250,S,0,1,1


In [5]:
pipe = build_pipeline(LogisticRegression(max_iter=2000))
for name, step in pipe.steps: print(f"{name:10s}->", type(step).__name__)
print()
for name, trans, cols in pipe.named_steps["prep"].transformers: print(f"{name:5s}", cols if isinstance(cols, list) else cols)

features  -> FunctionTransformer
prep      -> ColumnTransformer
model     -> LogisticRegression

num   ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']
cat   ['Sex', 'Embarked']
pass  ['Pclass', 'HasCabin', 'IsAlone']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipe.fit(X_train, y_train)
Xt = pipe.named_steps["prep"].transform(pipe.named_steps["features"].transform(X_train))
cols = pipe.named_steps["prep"].get_feature_names_out()
print("Matrix the model really sees:", Xt.shape)
pd.DataFrame(Xt[:5], columns=[c.split("__")[-1] for c in cols]).round(2)

Matrix the model really sees: (712, 12)


,Age,Fare,SibSp,Parch,FamilySize,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Pclass,HasCabin,IsAlone
0,-0.08,0.61,-0.47,-0.47,-0.56,1.0,0.0,0.0,1.0,3.0,0.0,1.0
1,-0.08,-0.74,-0.47,-0.47,-0.56,1.0,0.0,0.0,1.0,2.0,0.0,1.0
2,-0.08,4.54,-0.47,-0.47,-0.56,1.0,0.0,0.0,1.0,1.0,1.0,1.0
3,-0.89,-0.52,-0.47,0.73,0.07,0.0,0.0,0.0,1.0,3.0,0.0,0.0
4,0.11,-0.11,0.48,0.73,0.70,0.0,0.0,0.0,1.0,2.0,0.0,0.0


Text became numbers (`Sex`, `Embarked`), missing ages were filled with the training median, numeric columns were standardized, and every step was **learned from the training data only**, which prevents data leakage.

## 3. Model comparison (from `ml/train.py`)

In [7]:
m = json.loads((ROOT / "reports" / "metrics.json").read_text())
base = pd.DataFrame(m["cv_default"]).set_index("model").round(3)
tuned = pd.DataFrame(m["cv_tuned"]).set_index("model")
base["tuned_cv_roc_auc"] = tuned["cv_roc_auc"].round(3)
print("5-fold cross-validation on the training set:")
base

5-fold cross-validation on the training set:


,accuracy,f1,roc_auc,tuned_cv_roc_auc
model,,,,
Logistic Regression,0.798,0.733,0.859,0.859
Random Forest,0.815,0.755,0.870,0.882
Gradient Boosting,0.826,0.761,0.890,0.892


In [8]:
print("Selected (best tuned CV ROC-AUC):", m["model_name"])
print("Best parameters:", m["best_params"])
print("\nFinal test-set metrics:")
pd.Series(m["test_metrics"])

Selected (best tuned CV ROC-AUC): Gradient Boosting
Best parameters: {'subsample': 0.85, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1}

Final test-set metrics:


accuracy                             0.8101
precision                            0.7966
recall                               0.6812
f1                                   0.7344
roc_auc                              0.8201
tn                                  98.0000
fp                                  12.0000
fn                                  22.0000
tp                                  47.0000
majority_class_baseline_accuracy     0.6145
dtype: float64

In [9]:
pd.Series(m["permutation_importance"]).sort_values().plot.barh(figsize=(6,3.5), color="#2E5597", title="Permutation importance (test set)")
plt.xlabel("Drop in ROC-AUC when shuffled"); plt.tight_layout(); plt.savefig("/tmp/imp.png"); plt.show()

## 4. Model serialization: Pickle vs Joblib

In [10]:
model = joblib.load(ROOT / "models" / "titanic_pipeline.joblib")
tmp = Path(tempfile.mkdtemp())
t = time.perf_counter(); joblib.dump(model, tmp / "m.joblib"); tj = time.perf_counter() - t
t = time.perf_counter(); pickle.dump(model, open(tmp / "m.pkl", "wb")); tp = time.perf_counter() - t
print(f"joblib: {os.path.getsize(tmp/'m.joblib')/1024:.0f} KB, saved in {tj*1000:.1f} ms")
print(f"pickle: {os.path.getsize(tmp/'m.pkl')/1024:.0f} KB, saved in {tp*1000:.1f} ms")
a = joblib.load(tmp / "m.joblib").predict_proba(X_test)[:, 1]
b = pickle.load(open(tmp / "m.pkl", "rb")).predict_proba(X_test)[:, 1]
print("Same predictions after reloading:", np.allclose(a, b))

joblib: 265 KB, saved in 7.0 ms
pickle: 282 KB, saved in 4.1 ms


Same predictions after reloading: True


The saved object is the **whole pipeline** (feature engineering + preprocessing + model), so the API never has to repeat any preprocessing by hand.

## 5. Use the saved model on new passengers

In [11]:
new = pd.DataFrame([
  {"Pclass":1,"Sex":"female","Age":29,"SibSp":0,"Parch":0,"Fare":80,"Embarked":"S","HasCabin":1},
  {"Pclass":3,"Sex":"male","Age":30,"SibSp":0,"Parch":0,"Fare":7.25,"Embarked":"S","HasCabin":0},
  {"Pclass":3,"Sex":"male","Age":np.nan,"SibSp":1,"Parch":2,"Fare":20,"Embarked":"Q","HasCabin":0}])
new["P(survived)"] = model.predict_proba(new[RAW_FEATURES])[:, 1].round(3)
new

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,P(survived)
0,1,female,29.0,0,0,80.00,S,1,0.982
1,3,male,30.0,0,0,7.25,S,0,0.036
2,3,male,NaN,1,2,20.00,Q,0,0.226


## 6. Call the API in-process (same as `curl`)

In [12]:
from fastapi.testclient import TestClient
from app.main import app
with TestClient(app) as client:
    print(client.get("/health").json())
    r = client.post("/predict", json={"pclass":1,"sex":"female","age":29,"sibsp":0,"parch":0,"fare":80,"embarked":"S","has_cabin":True})
    print(r.status_code, r.json())
    r = client.post("/predict", json={"pclass":9,"sex":"male","sibsp":0,"parch":0,"fare":10,"embarked":"S"})
    print(r.status_code, r.json()["detail"][0]["msg"])

{'status': 'ok', 'model_loaded': True}
200 {'survived': True, 'survival_probability': 0.9821, 'prediction': 'Survived', 'confidence': 'high', 'model': 'Gradient Boosting'}
422 Input should be 1, 2 or 3


/usr/local/lib/python3.12/dist-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
